In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/age_size/inference'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Age-Size GPDF Dataset Inference

Fit GLM and GP posterior models to the generated age-size Soay sheep GP-IBM data. Survival, reproduction, and growth depend on size and age; recruitment depends on age; recruit size depends on maternal size; and sex probability is fixed outside inference. Age 0 is structurally excluded from reproduction fitting.
```python
sim_data.to_csv(project_dir / "IBMs/Age_Size/gp_ibm_dataset_agesize.csv", index=False)
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from jax import random
from numpyro.infer import MCMC, NUTS
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

jax.config.update("jax_enable_x64", True)

num_warmup = 1000
num_samples = 2500
num_chains = 2

project_dir = PROJECT_ROOT
dataset_path = GENERATED / 'soay/age_size/data/gp_ibm_dataset_agesize.csv'

In [ ]:
df = pd.read_csv(dataset_path)

survival = df[["z", "age", "Surv"]].dropna()
reproduction = df.loc[df["age"] >= 1, ["z", "age", "Repr"]].dropna()
growth = df[["z", "age", "z1"]].dropna()
recruitment = df[["age", "Recr"]].dropna()
recruit_size = df[["z", "Rcsz"]].dropna()

X_surv = survival[["z", "age"]].to_numpy(np.float64)
y_surv = survival["Surv"].to_numpy(np.float64)

X_repr = reproduction[["z", "age"]].to_numpy(np.float64)
y_repr = reproduction["Repr"].to_numpy(np.float64)

X_growth = growth[["z", "age"]].to_numpy(np.float64)
y_growth = growth["z1"].to_numpy(np.float64)

X_recr = recruitment[["age"]].to_numpy(np.float64)
y_recr = recruitment["Recr"].to_numpy(np.float64)

X_rcsz = recruit_size[["z"]].to_numpy(np.float64)
y_rcsz = recruit_size["Rcsz"].to_numpy(np.float64)

print("Survival", len(y_surv))
print("Reproduction", len(y_repr))
print("Growth", len(y_growth))
print("Recruitment", len(y_recr))
print("Recruit Size", len(y_rcsz))

In [ ]:
df.head()

In [ ]:
def rbf_ard(X1, X2, variance, lengthscales):
    scaled_difference = (X1[:, None, :] - X2[None, :, :]) / lengthscales
    squared_distance = jnp.sum(scaled_difference ** 2, axis=-1)
    return variance * jnp.exp(-0.5 * squared_distance)


def linear_binary(X, y):
    intercept = numpyro.sample("int", dist.Normal(0, 5))
    slope = numpyro.sample("slope", dist.Normal(jnp.zeros(X.shape[1]), 5).to_event(1))
    numpyro.sample("y", dist.Bernoulli(logits=intercept + X @ slope), obs=y)


def linear_gaussian(X, y):
    intercept = numpyro.sample("int", dist.Normal(0, 5))
    slope = numpyro.sample("slope", dist.Normal(jnp.zeros(X.shape[1]), 5).to_event(1))
    noise = numpyro.sample("noise", dist.HalfNormal(1))
    numpyro.sample("y", dist.Normal(intercept + X @ slope, noise), obs=y)


def gp_binary(X, y):
    intercept = numpyro.sample("int", dist.Normal(0, 5))
    variance = numpyro.sample("var", dist.HalfNormal(2))
    lengthscales = numpyro.sample("ls", dist.LogNormal(jnp.zeros(X.shape[1]), 1).to_event(1))
    latent = numpyro.sample("latent", dist.Normal(0, 1).expand([len(X)]).to_event(1))
    covariance = rbf_ard(X, X, variance, lengthscales) + 5e-4 * jnp.eye(len(X))
    mean = numpyro.deterministic("mu", jnp.linalg.cholesky(covariance) @ latent + intercept)
    numpyro.sample("y", dist.Bernoulli(logits=mean), obs=y)


def gp_gaussian(X, y):
    intercept = numpyro.sample("int", dist.Normal(0, 5))
    variance = numpyro.sample("var", dist.HalfNormal(2))
    lengthscales = numpyro.sample("ls", dist.LogNormal(jnp.zeros(X.shape[1]), 1).to_event(1))
    noise = numpyro.sample("noise", dist.HalfNormal(1))
    latent = numpyro.sample("latent", dist.Normal(0, 1).expand([len(X)]).to_event(1))
    covariance = rbf_ard(X, X, variance, lengthscales) + 5e-4 * jnp.eye(len(X))
    mean = numpyro.deterministic("mu", jnp.linalg.cholesky(covariance) @ latent + intercept)
    numpyro.sample("y", dist.Normal(mean, noise), obs=y)


def fit(model, key, **kwargs):
    mcmc = MCMC(NUTS(model), num_warmup=num_warmup, num_samples=num_samples, num_chains=num_chains)
    mcmc.run(key, **kwargs)
    mcmc.print_summary(exclude_deterministic=True)
    return mcmc.get_samples()

In [ ]:
rng = random.key(0)
keys = random.split(rng, 10)

In [ ]:
surv_glm = fit(linear_binary, keys[0], X=X_surv, y=y_surv)
repr_glm = fit(linear_binary, keys[1], X=X_repr, y=y_repr)
growth_glm = fit(linear_gaussian, keys[2], X=X_growth, y=y_growth)
recr_glm = fit(linear_binary, keys[3], X=X_recr, y=y_recr)
rcsz_glm = fit(linear_gaussian, keys[4], X=X_rcsz, y=y_rcsz)

In [ ]:
surv_gp = fit(gp_binary, keys[5], X=X_surv, y=y_surv)

In [ ]:
repr_gp = fit(gp_binary, keys[6], X=X_repr, y=y_repr)

In [ ]:
growth_gp = fit(gp_gaussian, keys[7], X=X_growth, y=y_growth)

In [ ]:
recr_gp = fit(gp_binary, keys[8], X=X_recr, y=y_recr)

In [ ]:
rcsz_gp = fit(gp_gaussian, keys[9], X=X_rcsz, y=y_rcsz)

In [ ]:
gpdf_agesize_glm_samples = {
    "surv_int": surv_glm["int"],
    "surv_slope": surv_glm["slope"][:, 0],
    "surv_age": surv_glm["slope"][:, 1],
    "repr_int": repr_glm["int"],
    "repr_slope": repr_glm["slope"][:, 0],
    "repr_age": repr_glm["slope"][:, 1],
    "growth_int": growth_glm["int"],
    "growth_slope": growth_glm["slope"][:, 0],
    "growth_age": growth_glm["slope"][:, 1],
    "growth_noise": growth_glm["noise"],
    "recr_int": recr_glm["int"],
    "recr_age": recr_glm["slope"][:, 0],
    "rcsz_int": rcsz_glm["int"],
    "rcsz_slope": rcsz_glm["slope"][:, 0],
    "rcsz_noise": rcsz_glm["noise"],
}

gpdf_agesize_gp_samples = {
    "surv_int": surv_gp["int"],
    "surv_var": surv_gp["var"],
    "surv_ls": surv_gp["ls"],
    "surv_mu": surv_gp["mu"],
    "surv_latent": surv_gp["latent"],
    "repr_int": repr_gp["int"],
    "repr_var": repr_gp["var"],
    "repr_ls": repr_gp["ls"],
    "repr_mu": repr_gp["mu"],
    "repr_latent": repr_gp["latent"],
    "growth_int": growth_gp["int"],
    "growth_var": growth_gp["var"],
    "growth_ls": growth_gp["ls"],
    "growth_noise": growth_gp["noise"],
    "growth_mu": growth_gp["mu"],
    "growth_latent": growth_gp["latent"],
    "recr_int": recr_gp["int"],
    "recr_var": recr_gp["var"],
    "recr_ls": recr_gp["ls"],
    "recr_mu": recr_gp["mu"],
    "recr_latent": recr_gp["latent"],
    "rcsz_int": rcsz_gp["int"],
    "rcsz_var": rcsz_gp["var"],
    "rcsz_ls": rcsz_gp["ls"],
    "rcsz_noise": rcsz_gp["noise"],
    "rcsz_mu": rcsz_gp["mu"],
    "rcsz_latent": rcsz_gp["latent"],
}

np.savez_compressed(
    "gpdf_agesize_glm_samples.npz",
    **{name: np.asarray(jax.device_get(value)) for name, value in gpdf_agesize_glm_samples.items()},
)

np.savez_compressed(
    "gpdf_agesize_gp_samples.npz",
    **{name: np.asarray(jax.device_get(value)) for name, value in gpdf_agesize_gp_samples.items()},
)